# indian_equities_strat_004

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (150).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 14 |
| Detected functions | kalman_filter_optimized, apply_kalman_trend_strategy_vectorbt, backtest_with_sl_tp, process_multiple_tickers_optimized, run_robustness_testing_with_sl_tp, save_results_to_csv |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install numpy pandas matplotlib plotly numba vectorbt


In [ ]:
import pandas as pd

path = 'drive/My Drive/ipynb'


In [ ]:

# Use a raw string or replace backslashes with forward slashes
data = pd.read_parquet(path+'/AUG/AUG_2024.parquet')  # Raw string

# Display the DataFrame
data

In [ ]:
data.Ticker.unique()

In [ ]:

# Filter the dataset to include only the specified 10 tickers
data = data[data['Ticker'].isin(['ASIANPAINT.NSE', 'BAJFINANCE.NSE', 'BHARTIARTL.NSE', 'LT.NSE', 'COALINDIA.NSE', 'TCS.NSE', 'MARUTI.NSE',
                                 'RELIANCE.NSE', 'HDFCBANK.NSE', 'SBIN.NSE', 'HINDUNILVR.NSE', 'ITC.NSE', 'TATASTEEL.NSE', 'ULTRACEMCO.NSE',
                                 'KOTAKBANK.NSE', ])]

# Display the filtered DataFrame
data


In [ ]:
# Ensure Date_Time column is in datetime format
data['Date_Time'] = pd.to_datetime(data['Date_Time'])

# Filter rows where the time is between 09:16 and 09:30
data = data[(data['Date_Time'].dt.time >= pd.to_datetime('09:20').time()) &
                     (data['Date_Time'].dt.time <= pd.to_datetime('09:31').time())]

# Display the filtered DataFrame
data


In [ ]:
data.Ticker.unique()

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
from numba import njit
from tqdm import tqdm

# Optimized Kalman Filter Function with Numba for Speed
@njit
def kalman_filter_optimized(prices, process_variance, measurement_variance):
    n = len(prices)
    estimates = np.empty(n)
    estimates.fill(np.nan)
    estimate = prices[0]
    error_covariance = 1.0

    for t in range(n):
        prediction = estimate
        prediction_covariance = error_covariance + process_variance

        kalman_gain = prediction_covariance / (prediction_covariance + measurement_variance)
        estimate = prediction + kalman_gain * (prices[t] - prediction)
        error_covariance = (1 - kalman_gain) * prediction_covariance

        estimates[t] = estimate

    return estimates

# Apply Kalman Filter and Generate Buy/Sell Signals
def apply_kalman_trend_strategy_vectorbt(data, process_variance, measurement_variance):
    prices = data['LTP'].to_numpy()
    filtered_prices = kalman_filter_optimized(prices, process_variance, measurement_variance)

    data['FilteredPrice'] = filtered_prices

    # Generate Buy and Sell signals
    data['BuySignal'] = (data['LTP'].shift(1) <= data['FilteredPrice'].shift(1)) & (data['LTP'] > data['FilteredPrice'])
    data['SellSignal'] = (data['LTP'].shift(1) >= data['FilteredPrice'].shift(1)) & (data['LTP'] < data['FilteredPrice'])

    return data

# Backtest with Stop-Loss and Take-Profit
def backtest_with_sl_tp(data, initial_capital, brokerage, sl_percent, tp_percent):
    close = data['LTP'].to_numpy()
    buy_signal = data['BuySignal'].to_numpy()
    sell_signal = data['SellSignal'].to_numpy()

    # Portfolio simulation with stop-loss and take-profit
    pf = vbt.Portfolio.from_signals(
        close=close,
        entries=buy_signal,
        exits=sell_signal,
        sl_stop=sl_percent / 100,
        tp_stop=tp_percent / 100,
        init_cash=initial_capital,
        fees=brokerage
    )

    # Extract trade records
    trade_log = pf.trades.records
    trade_log['PnL'] = trade_log['exit_price'] - trade_log['entry_price']
    trade_log['Cumulative PnL'] = trade_log['PnL'].cumsum()  # Track cumulative PnL

    # Calculate cumulative profit
    cumulative_profit = trade_log['Cumulative PnL'].iloc[-1] if not trade_log.empty else 0

    return cumulative_profit

# Process Multiple Tickers for Robustness Testing
def process_multiple_tickers_optimized(data, initial_capital, brokerage, process_variance, measurement_variance, sl_percent, tp_percent):
    grouped_data = data.groupby('Ticker')
    trade_logs = []

    # Parallel processing of tickers
    for ticker, group_data in tqdm(grouped_data, desc="Processing tickers", ncols=100):
        group_data = apply_kalman_trend_strategy_vectorbt(group_data, process_variance, measurement_variance)
        cumulative_profit = backtest_with_sl_tp(group_data, initial_capital, brokerage, sl_percent, tp_percent)

        # Print the results for each parameter combination
        print(f"Process Variance: {process_variance}, Measurement Variance: {measurement_variance}, SL: {sl_percent}, TP: {tp_percent}, Cumulative Profit: {cumulative_profit}")

        # Add metadata to the trade log
        trade_log = {
            'Ticker': ticker,
            'Process Variance': process_variance,
            'Measurement Variance': measurement_variance,
            'SL Percent': sl_percent,
            'TP Percent': tp_percent,
            'Cumulative Profit': cumulative_profit
        }
        trade_logs.append(trade_log)

    return trade_logs

# Run Robustness Testing with SL and TP
def run_robustness_testing_with_sl_tp(data, initial_capital, brokerage, process_variance_values, measurement_variance_values, sl_percent_values, tp_percent_values, output_file):
    all_trade_logs = []

    # Iterate over parameter grids with reduced nesting
    for process_variance in tqdm(process_variance_values, desc="Process Variance", ncols=100):
        for measurement_variance in measurement_variance_values:
            for sl_percent in sl_percent_values:
                for tp_percent in tp_percent_values:
                    trade_logs = process_multiple_tickers_optimized(
                        data, initial_capital, brokerage, process_variance, measurement_variance, sl_percent, tp_percent
                    )
                    all_trade_logs.extend(trade_logs)

    # Save results to CSV
    all_trade_logs_df = pd.DataFrame(all_trade_logs)
    all_trade_logs_df.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")

if __name__ == "__main__":
    # Parameters
    initial_capital = 100000  # Initial capital for backtesting
    brokerage = 0.001  # Constant brokerage fee
    process_variance_values = [0.1, 1, 3, 5, 7, 10, 14, 17, 21]  # Process variance values to test
    measurement_variance_values = [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.5]  # Measurement variance values to test
    sl_percent_values = [0.5, 1.0, 1.5, 2.0]  # Stop loss values to test
    tp_percent_values = [1.0, 1.5, 2.0, 2.5, 3.0]  # Take profit values to test
    output_file = "trade_logs_with_sl_tp.csv"  # Output file path

    # Convert the 'datetime' column to datetime format and set it as the index
    data['datetime'] = pd.to_datetime(data['Date_Time'])
    data.set_index('datetime', inplace=True)

    # Drop duplicates if any, keeping the first occurrence (or you can aggregate them based on a rule)
    data = data.loc[~data.index.duplicated(keep='first')]

    # If you need to resample the data (e.g., to 1-minute intervals), use the resample method
    # For example, resample to 1-minute frequency:
    data_resampled = data.resample('1T').last()  # Resampling to 1-minute interval, you can change '1T' as needed

    # Proceed with robustness testing
    run_robustness_testing_with_sl_tp(
        data_resampled, initial_capital, brokerage, process_variance_values, measurement_variance_values,
        sl_percent_values, tp_percent_values, output_file
    )


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Function to process and save the results to CSV
def save_results_to_csv(data, initial_capital, brokerage, process_variance_values, measurement_variance_values, sl_percent_values, tp_percent_values, output_file):
    all_trade_logs = []

    # Iterate over parameter grids and run backtesting for each combination
    for process_variance in tqdm(process_variance_values, desc="Process Variance", ncols=100):
        for measurement_variance in measurement_variance_values:
            for sl_percent in sl_percent_values:
                for tp_percent in tp_percent_values:
                    cumulative_profit = backtest_with_sl_tp(
                        data, initial_capital, brokerage, process_variance, measurement_variance, sl_percent, tp_percent
                    )

                    # Create a trade log dictionary for each combination
                    trade_log = {
                        'Process Variance': process_variance,
                        'Measurement Variance': measurement_variance,
                        'SL Percent': sl_percent,
                        'TP Percent': tp_percent,
                        'Cumulative Profit': cumulative_profit
                    }
                    all_trade_logs.append(trade_log)

    # Save results to a CSV file
    all_trade_logs_df = pd.DataFrame(all_trade_logs)
    all_trade_logs_df.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")

# Example to run the function
if __name__ == "__main__":
    # Define the parameters and file path
    initial_capital = 100000  # Initial capital for backtesting
    brokerage = 0.001  # Constant brokerage fee
    process_variance_values = [0.1, 1, 3, 5, 7, 10, 14, 17, 21]  # Process variance values to test
    measurement_variance_values = [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.5]  # Measurement variance values to test
    sl_percent_values = [0.5, 1.0, 1.5, 2.0]  # Stop loss values to test
    tp_percent_values = [1.0, 1.5, 2.0, 2.5, 3.0]  # Take profit values to test
    output_file = "backtest_results.csv"  # Output file path

    # Assume 'data' is already loaded as a DataFrame
    save_results_to_csv(
        data, initial_capital, brokerage, process_variance_values, measurement_variance_values,
        sl_percent_values, tp_percent_values, output_file
    )


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Assuming the data is in a pandas DataFrame called df
df = pd.read_csv('optimized_trades_with_brokerage.csv')

# Convert 'entry_time' and 'exit_time' to datetime, auto-detect the format
df['entry_time'] = pd.to_datetime(df['entry_time'], errors='coerce').dt.time
df['exit_time'] = pd.to_datetime(df['exit_time'], errors='coerce').dt.time

# Plot 1: Cumulative profit over time for each ticker
fig1 = go.Figure()
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]
    fig1.add_trace(go.Scatter(x=ticker_data['exit_time'],
                             y=ticker_data['Cumulative_PnL'],
                             mode='lines',
                             name=ticker))
fig1.update_layout(title='Cumulative Profit Over Time by Ticker',
                  xaxis_title='Time',
                  yaxis_title='Cumulative PnL ($)',
                  template='plotly_dark',
                  width=1200, height=600)  # Increased width
fig1.show()

# Plot 2: Drawdown over time for each ticker
fig2 = go.Figure()
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]
    ticker_data['Portfolio_Value'] = ticker_data['Cumulative_PnL'] + 100000  # Assuming initial capital of $100,000
    ticker_data['drawdown'] = ticker_data['Portfolio_Value'] - ticker_data['Portfolio_Value'].cummax()
    fig2.add_trace(go.Scatter(x=ticker_data['exit_time'],
                             y=ticker_data['drawdown'],
                             mode='lines',
                             name=ticker))
fig2.update_layout(title='Drawdown Over Time by Ticker',
                  xaxis_title='Time',
                  yaxis_title='Drawdown ($)',
                  template='plotly_dark',
                  width=1200, height=600)  # Increased width
fig2.show()

# Plot 3: Total Profit Percentage by Ticker (Bar chart)
total_profit_percent_by_ticker = []

# Calculate total profit percentage for each ticker
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]
    total_profit_percent = (ticker_data['PnL_Percent'].sum())  # Sum of profit percentage
    total_profit_percent_by_ticker.append(total_profit_percent)

fig3 = px.bar(x=df['Ticker'].unique(), y=total_profit_percent_by_ticker,
              labels={'x': 'Ticker', 'y': 'Total Profit Percentage (%)'},
              title='Total Profit Percentage by Ticker',
              template='plotly_dark')
fig3.update_traces(marker=dict(color='skyblue', opacity=0.7))
fig3.update_layout(width=1200, height=600)  # Increased width
fig3.show()

# Plot 4: PnL distribution (Histogram)
fig4 = px.histogram(df, x='PnL', nbins=20, title='PnL Distribution',
                    labels={'PnL': 'PnL ($)'},
                    template='plotly_dark')
fig4.update_traces(marker=dict(color='purple', opacity=0.75))
fig4.update_layout(width=1200, height=600)  # Increased width
fig4.show()

# Plot 5: Scatter plot of entry and exit times (with colors for entry and exit)
fig5 = go.Figure()

# Loop through each ticker
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]

    # Format the entry and exit times to HH:MM:SS:MS
    ticker_data['entry_time_str'] = ticker_data['entry_time'].astype(str)
    ticker_data['exit_time_str'] = ticker_data['exit_time'].astype(str)

    # Entry time (x-axis, blue color)
    fig5.add_trace(go.Scatter(x=ticker_data['entry_time_str'],
                             y=ticker_data['exit_time_str'],
                             mode='markers',
                             name=f'{ticker} Entry Time',
                             marker=dict(color='blue', size=8, opacity=0.6),
                             text=ticker_data['entry_time_str']))

    # Exit time (x-axis, red color)
    fig5.add_trace(go.Scatter(x=ticker_data['exit_time_str'],
                             y=ticker_data['exit_time_str'],
                             mode='markers',
                             name=f'{ticker} Exit Time',
                             marker=dict(color='red', size=8, opacity=0.6),
                             text=ticker_data['exit_time_str']))

fig5.update_layout(
    title='Entry Time vs Exit Time',
    xaxis_title='Entry Time (HH:MM:SS)',
    yaxis_title='Exit Time (HH:MM:SS)',
    template='plotly_dark',
    showlegend=True,
    width=1200,
    height=600  # Increased width
)

fig5.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Assuming the data is in a pandas DataFrame called df
df = pd.read_csv('optimized_trades_with_brokerage.csv')

# Create a PDF file to save the plots
with PdfPages('trading_plots.pdf') as pdf:

    # Plotting 'Cumulative_PnL' column
    plt.figure(figsize=(12, 6))
    plt.plot(df['Cumulative_PnL'])
    plt.title('Cumulative Profit and Loss Over Time')
    plt.xlabel('Time')
    plt.ylabel('Cumulative PnL ($)')
    pdf.savefig()  # Save the current figure to the PDF
    plt.close()

    plt.show()
    # Calculate total profit percentage for each ticker
    total_profit_percent_by_ticker = []
    for ticker in df['Ticker'].unique():
        ticker_data = df[df['Ticker'] == ticker]
        total_profit_percent = ticker_data['PnL_Percent'].sum()  # Sum of profit percentage
        total_profit_percent_by_ticker.append(total_profit_percent)

    # Convert to a DataFrame for sorting and better control
    ticker_summary = pd.DataFrame({
        'Ticker': df['Ticker'].unique(),
        'Total_Profit_Percent': total_profit_percent_by_ticker
    }).sort_values(by='Total_Profit_Percent', ascending=False)

    # Create horizontal bar chart with thinner bars
    plt.figure(figsize=(14, 24))  # Adjusted height for 50 tickers
    plt.barh(ticker_summary['Ticker'], ticker_summary['Total_Profit_Percent'], color='skyblue', alpha=0.7, height=0.6)
    plt.title('Total Profit Percentage by Ticker', fontsize=16)
    plt.xlabel('Total Profit Percentage (%)', fontsize=12)
    plt.ylabel('Ticker', fontsize=12)
    plt.tight_layout()
    pdf.savefig()  # Save the figure
    plt.close()
    plt.show()
